In [0]:
from pyspark.sql.functions import col, to_timestamp, lag, round
from pyspark.sql.window import Window

history_df = spark.table("workspace.default.bronze_quality_history")

display(history_df)

In [0]:
history_clean = (
    history_df
    .select(
        "run_timestamp",
        "table_name",
        col("quality_score").cast("double").alias("quality_score")
    )
)

display(history_clean)

In [0]:
window_spec = Window.partitionBy("table_name").orderBy("run_timestamp")

trend_df = (
    history_clean
    .withColumn("previous_quality_score", lag("quality_score").over(window_spec))
    .withColumn(
        "quality_score_change",
        round(col("quality_score") - col("previous_quality_score"), 2)
    )
)

display(trend_df)

In [0]:
trend_df.write.mode("overwrite").saveAsTable(
    "workspace.default.gold_quality_trends"
)

In [0]:
def get_most_degraded_table():
    row = (
        spark.table("workspace.default.gold_quality_trends")
        .filter(col("quality_score_change").isNotNull())
        .orderBy(col("quality_score_change").asc())
        .limit(1)
        .collect()[0]
    )

    return {
        "table_name": row["table_name"],
        "run_timestamp": row["run_timestamp"],
        "previous_quality_score": row["previous_quality_score"],
        "current_quality_score": row["quality_score"],
        "quality_score_change": row["quality_score_change"]
    }


def build_trend_agent_response(user_question):
    result = get_most_degraded_table()

    response = f"""
Question:
{user_question}

Answer:
The table with the highest quality degradation is `{result["table_name"]}`.

Trend Evidence:
- Previous Quality Score: {result["previous_quality_score"]}
- Current Quality Score: {result["current_quality_score"]}
- Quality Score Change: {result["quality_score_change"]}
- Run Timestamp: {result["run_timestamp"]}

Interpretation:
A negative quality score change indicates degradation in data quality.
This table should be prioritized for investigation.

Recommended Actions:
- Compare failed rules between the previous and current run.
- Check whether new null, duplicate, schema, or accepted-value failures appeared.
- Review recent ingestion or transformation changes.
- Add alerting when quality score drops beyond an agreed threshold.
- Track recurring degradation patterns over time.

Agent Tools Used:
1. get_most_degraded_table()
2. gold_quality_trends Delta table
"""

    return response


print(
    build_trend_agent_response(
        "Which table quality degraded the most over time?"
    )
)